In [12]:
# 1. Instalar biblioteca para cálculo das métricas de legibilidade tradicionais
!pip install textstat -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 22.7 MB/s eta 0:00:00


In [20]:
import torch
import polars as pl
import numpy as np
import textstat
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Configurações de exibição do Polars para garantir que nada seja truncado
pl.Config.set_tbl_rows(100)
pl.Config.set_fmt_str_lengths(150)
pl.Config.set_tbl_cols(10)

# 2. Definição do Corpus de Teste (Frases Simples vs Complexas)
novas_frases = [
    {"doc_id": 1, "nivel": "Nível 1 (Básico)", "texto": "The small dog, owned by Zachary, barked loudly at the blue mailbox."},
    {"doc_id": 2, "nivel": "Nível 2 (Médio)", "texto": "The company expects to increase its quarterly revenue by implementing new strategic plans developed during the annual board meetings."},
    {"doc_id": 3, "nivel": "Nível 1 (Básico)", "texto": "An old owl screeched inside a hollow tree branch during the dark and chilly night."},
    {"doc_id": 4, "nivel": "Nível 3 (Avançado)", "texto": "The rapid development of artificial intelligence has revolutionized modern software engineering paradigms and research methodologies."},
    {"doc_id": 5, "nivel": "Nível 2 (Médio)", "texto": "We need to immediately optimize our computational framework to bypass severe operational bottlenecks."},
    {"doc_id": 6, "nivel": "Nível 3 (Avançado)", "texto": "The epistemological paradigms of modern philosophical thoughts present profound cognitive challenges for graduate students."}
]

# 3. Inicialização do Modelo e Tokenizer (distilgpt2)
modelo_nome = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(modelo_nome)
model = AutoModelForCausalLM.from_pretrained(modelo_nome)
model.eval()

resultados_completos = []

# 4. Processamento e extração de métricas de base
with torch.no_grad():
    for item in novas_frases:
        doc_id = item["doc_id"]
        nivel = item["nivel"]
        texto = item["texto"]

        # Cálculo da Perplexidade com distilgpt2
        inputs = tokenizer(texto, return_tensors="pt")
        input_ids = inputs["input_ids"]
        outputs = model(input_ids, labels=input_ids)
        loss_global = outputs.loss.item()
        perplexidade = round(np.exp(loss_global), 2)

        # Cálculo de Métricas Tradicionais de Legibilidade (Flesch)
        fk_grade = round(textstat.flesch_kincaid_grade(texto), 2)
        reading_ease = round(textstat.flesch_reading_ease(texto), 2)
        syllable_count = textstat.syllable_count(texto)
        lexicon_count = textstat.lexicon_count(texto)

        resultados_completos.append({
            "doc_id": doc_id,
            "Nível Real": nivel,
            "Texto": texto,
            "Perplexidade (GPT2 - Falha)": perplexidade,
            "Flesch-Kincaid (Ano Escolar)": fk_grade,
            "Facilidade de Leitura (0-100)": reading_ease,
            "Qtd Palavras": lexicon_count,
            "Qtd Sílabas": syllable_count
        })

# Criando o DataFrame base do Polars
df_base = pl.DataFrame(resultados_completos)

# 5. Cálculo do Índice Híbrido (0-100)
ppl_vals = df_base["Perplexidade (GPT2 - Falha)"].to_numpy()
fk_vals = df_base["Flesch-Kincaid (Ano Escolar)"].to_numpy()

# Função de Normalização Min-Max para colocá-los no mesmo patamar de escala
def min_max_scale(arr):
    denom = arr.max() - arr.min()
    if denom == 0:
        return np.zeros_like(arr)
    return (arr - arr.min()) / denom

ppl_norm = min_max_scale(ppl_vals)
fk_norm = min_max_scale(fk_vals)

# Combinação ponderada (60% Peso Humano-Estrutural, 40% IA-Fluidez)
peso_flesch = 0.6
peso_ppl = 0.4
ind_hibrido = (fk_norm * peso_flesch) + (ppl_norm * peso_ppl)
ind_hibrido_100 = np.round(ind_hibrido * 100, 2)

# 6. Adição da coluna final e ordenação lógica por dificuldade real combinada
df_final_consolidado = df_base.with_columns([
    pl.Series("Índice Híbrido (0-100)", ind_hibrido_100)
]).sort("Índice Híbrido (0-100)")

print("=== TABELA CONSOLIDADA COMPLETA (ORDENADA PELO ÍNDICE HÍBRIDO) ===")
display(df_final_consolidado)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

=== TABELA CONSOLIDADA COMPLETA (ORDENADA PELO ÍNDICE HÍBRIDO) ===


doc_id,Nível Real,Texto,Perplexidade (GPT2 - Falha),Flesch-Kincaid (Ano Escolar),Facilidade de Leitura (0-100),Qtd Palavras,Qtd Sílabas,Índice Híbrido (0-100)
i64,str,str,f64,f64,f64,i64,i64,f64
1,"""Nível 1 (Básico)""","""The small dog, owned by Zachary, barked loudly at the blue mailbox.""",113.45,4.82,81.86,12,16,17.98
3,"""Nível 1 (Básico)""","""An old owl screeched inside a hollow tree branch during the dark and chilly night.""",185.64,5.21,84.45,15,19,41.32
2,"""Nível 2 (Médio)""","""The company expects to increase its quarterly revenue by implementing new strategic plans developed during the annual board meetings.""",104.39,15.42,18.35,19,38,51.17
4,"""Nível 3 (Avançado)""","""The rapid development of artificial intelligence has revolutionized modern software engineering paradigms and research methodologies.""",54.48,22.51,-39.63,15,41,60.0
6,"""Nível 3 (Avançado)""","""The epistemological paradigms of modern philosophical thoughts present profound cognitive challenges for graduate students.""",95.4,19.37,-18.88,14,35,61.83
5,"""Nível 2 (Médio)""","""We need to immediately optimize our computational framework to bypass severe operational bottlenecks.""",127.1,19.43,-21.11,13,33,71.7
